
# ORF307: Homework 6

In [13]:
import pandas as pd
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

# Question 3

Consider the following LP:
$$
\begin{array}{ll}\tag{3}
\text{minimize} &13x_1 + 10x_2 + 6x_3\\
\text{subject to} &5x_1 + x_2 + 3x_3 = 8\\
& 3x_1 + x_2 = 3\\
& x_1, x_2, x_3 \ge 0.
\end{array}
$$

In [14]:
'''
This code is provided to help with question 4.
This code returns optimal primal variables x 
and dual variables y.
'''
import cvxpy as cp
import numpy as np
import numpy.linalg as la

def simplex_iteration(x, B, problem):
    """Perform one simplex iteration given 
    - basic feasible solution x
    - basis B
    
    It returns new x, new basis, new dual variable,
    and termination flag (true/false)
    """
    A, b, c = problem['A'], problem['b'], problem['c']
    m, n = A.shape
    A_B = A[:, B]
    
    # Compute reduced cost vector
    p = la.solve(A_B.T, c[B])
    c_bar = c - A.T @ p
    
    # Check optimality
    if np.all(c_bar >= 0):
        print("Optimal solution found!")
        return x, B, -p, True

    # Choose j such that c_bar < 0 (first one)
    j = np.where(c_bar < 0)[0][0]
    
    # Compute search direction d
    d = np.zeros(n)
    d[j] = 1
    d[B] = la.solve(A_B, -A[:, j])
    
    # Check for unboundedness
    if np.all(d >= 0):
        print("Unbounded problem!")
        return None, None, True
        
    # Compute step length theta
    d_i = np.where(d[B] < 0)[0]
    theta = np.min(- x[B[d_i]] / d[B[d_i]])
    i = B[d_i[np.argmin(- x[B[d_i]] / d[B[d_i]])]]
    
    # Compute next point
    x_next = x + theta * d
    
    # Compute next basis
    B_next = B
    B_next[np.where(B == i)[0]] = j
  
    return x_next, B_next, -p, False

def simplex_algorithm(x, B, problem, max_iter=1000):
    """Run simplex algorithm"""

    for k in range(max_iter):
            
        x, B, y, end = simplex_iteration(x, B, problem)
        
        if end:
            break
    return x, B, y

(a) Solve it using the big-M formulation as in Q3 obtaining optimal primal and dual variables (use the provided function).

In [15]:
M = 10000

A = np.array([
  [5., 1., 3., 1., 0.],
  [3., 1., 0., 0., 1.]
]
)

b = np.array([8., 3.])
c = np.array([13., 10., 6., M, M])

problem = {"A": A, "b": b, 'c': c}

init_x = np.array([0., 0., 0., 8., 3.])
init_basis = np.array([3, 4], dtype=int)

opt_x, opt_basis, opt_dual = simplex_algorithm(init_x, init_basis, problem)

x1_opt, x2_opt, x3_opt = opt_x[:3]

objective_val = 13 * x1_opt + 10 * x2_opt + 6 * x3_opt

print("\n========== Optimal Results for Original LP (3) ==========")
print(f"x* = ({x1_opt:.4f}, {x2_opt:.4f}, {x3_opt:.4f})")
print(f"Optimal Objective Value : {objective_val:.4f}")
print("----------------------------------------------------------")
print("Dual Variables (y*):", opt_dual)
print("==========================================================")

Optimal solution found!

========== Optimal Results for Original LP (3) ==========
x* = (1.0000, 0.0000, 1.0000)
Optimal Objective Value : 19.0000
----------------------------------------------------------
Dual Variables (y*): [-2. -1.]


(c) Solve the dual using CVXPY and compare the optimal primal-dual variables with the ones from (a).

In [22]:
dual_y1 = cp.Variable()
dual_y2 = cp.Variable()

dual_constraints = [
    5 * dual_y1 + 3 * dual_y2 + 13 >= 0,
    dual_y1 + dual_y2 + 10 >= 0,
    3 * dual_y1 + 6 >= 0
]

dual_objective = cp.Minimize(8 * dual_y1 + 3 * dual_y2)

dual_problem = cp.Problem(dual_objective, dual_constraints)
dual_problem.solve()

print("\n===== Dual Problem Results (CVXPY) =====")
print(f"Optimal y1 = {dual_y1.value:.4f}")
print(f"Optimal y2 = {dual_y2.value:.4f}")
print(f"Dual Objective Value (-b^T y) = {-8 * dual_y1.value - 3 * dual_y2.value:.4f}")
print("=========================================")
print("Note that the answers are the same as part A where we used the Big M formation. " \
"This is because, while it is an alternate formation of the problem, the inherent problem"
" is the same and thus the solution space will be the same")


===== Dual Problem Results (CVXPY) =====
Optimal y1 = -2.0000
Optimal y2 = -1.0000
Dual Objective Value (-b^T y) = 19.0000
Note that the answers are the same as part A where we used the Big M formation. This is because, while it is an alternate formation of the problem, the inherent problem is the same and thus the solution space will be the same
